In [5]:
import pandas as pd
from qdrant_client import QdrantClient
import os
from sentence_transformers import SentenceTransformer
from qdrant_client.http import models


In [2]:

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2254.77it/s]


In [12]:
# ger environment variables for Qdrant
QDRANT_URL = os.environ.get("QDRANT_URL", "")
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY", "")

#create qdrant client
qdrant_client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

In [13]:

#Method Check If collection exists in Qdrant if not create Collection
def collection_exists( qdrant_client: QdrantClient, collection_name: str, vectorsDimensions: int) -> bool:
    try:
        # Check if collection exists, create if not
        if not qdrant_client.collection_exists(collection_name=collection_name):
            qdrant_client.create_collection(
                collection_name=collection_name,
                vectors_config=models.VectorParams(
                                        size=vectorsDimensions,
                                        distance=models.Distance.COSINE),
            )
            #print(f"Collection '{collection_name}' created.")
        #else:
            #print(f"Collection '{collection_name}' already exists.")

    except Exception as e:
        print(f"Error creating or checking collection '{collection_name}': {e}")
        return False


In [21]:
qaDoc= pd.read_csv("../Dataset/coverwise_policy_qa_risk_testset_60.csv")

In [24]:
# Generate QA Pair text , encode them  and save into quadrant

#qaDoc= pd.read_csv("../Dataset/coverwise_policy_qa_risk_testset_60.json")

QA_PAIR_COLLECTION="insurancePolicy_QA_Pairs"
# Extract questions and answers
qa_text=[]
for _, row in qaDoc.iterrows():
    text = f"""category : {row['question_category']}
               question: {row['user_question']}
               expected Answer: {row['expected_answer_guidance']}
             """
    qa_text.append(text)

# Encode the questions and answers
print("\nEncoding questions and Answers...")
qa_vectors = embedding_model.encode(qa_text)

# Store encoded QA pairs with their original data
points = []
for i, row in qaDoc.iterrows():
    payload = {
        "id": row["test_id"],
        "category": row["question_category"],
        "question": row["user_question"],
        "answer": row["expected_answer_guidance"],
        "target_clause_or_feature": row["target_clause_or_feature"],
    }

    points.append(
        models.PointStruct(
            id=i,
            vector=qa_vectors[i].tolist(), # Convert numpy array to list for storage
            payload=payload
        )
    )


try:
       #Check if collection exists
    collection_exists(qdrant_client, QA_PAIR_COLLECTION, qa_vectors.shape[1])

        # save QA_payload in qdrant
    operation_info = qdrant_client.upsert(
           collection_name=QA_PAIR_COLLECTION,
            wait=True,
            points=points )
    print(f"Upsert operation info: {operation_info}")
    print(f"Successfully saved {len(points)} embeddings to Qdrant collection '{QA_PAIR_COLLECTION}'.")

except Exception as e:
        print(f"Error upserting vectors to collection '{QA_PAIR_COLLECTION}': {e}")


The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.

Encoding questions and Answers...
Upsert operation info: operation_id=1 status=<UpdateStatus.COMPLETED: 'completed'>
Successfully saved 60 embeddings to Qdrant collection 'insurancePolicy_QA_Pairs'.
